<a href="https://colab.research.google.com/github/ruicatzzz/aigc-detector/blob/daphne/notebooks/colab_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone -b daphne https://github.com/ruicatzzz/aigc-detector.git

Cloning into 'aigc-detector'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 121 (delta 42), reused 95 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 127.39 KiB | 852.00 KiB/s, done.
Resolving deltas: 100% (42/42), done.


In [2]:
%cd aigc-detector

/content/aigc-detector


In [6]:
!ls -la

total 52
drwxr-xr-x 7 root root  4096 Aug 29 13:07 .
drwxr-xr-x 1 root root  4096 Aug 29 13:07 ..
drwxr-xr-x 2 root root  4096 Aug 29 13:07 checkpoints
drwxr-xr-x 2 root root  4096 Aug 29 13:07 docs
-rw-r--r-- 1 root root 10244 Aug 29 13:07 .DS_Store
drwxr-xr-x 8 root root  4096 Aug 29 13:07 .git
-rw-r--r-- 1 root root     6 Aug 29 13:07 .gitignore
drwxr-xr-x 3 root root  4096 Aug 29 13:07 outputs
-rw-r--r-- 1 root root  2243 Aug 29 13:07 README.md
-rw-r--r-- 1 root root   297 Aug 29 13:07 requirements.txt
drwxr-xr-x 3 root root  4096 Aug 29 13:07 src


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import kagglehub
path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
print(path)

Using Colab cache for faster access to the 'cifake-real-and-ai-generated-synthetic-images' dataset.
/kaggle/input/cifake-real-and-ai-generated-synthetic-images


In [5]:
import shutil, os

# path is whatever printed from the previous cell
os.makedirs('data/cifake', exist_ok=True)
shutil.copytree(path, 'data/cifake', dirs_exist_ok=True)

'data/cifake'

In [18]:
!python -m src.train --data_dir data/cifake/train --epochs 5 --out checkpoints/cnn_cifake.pt

Using device: cuda
Classes found: {'FAKE': 0, 'REAL': 1}  (expect something like FAKE=0, REAL=1 — see docstring)
Epoch 1/5: 100% 704/704 [00:37<00:00, 18.85it/s]
Epoch 1: train_loss=0.2279 val_acc=0.9270
Epoch 2/5: 100% 704/704 [00:35<00:00, 19.80it/s]
Epoch 2: train_loss=0.1611 val_acc=0.9444
Epoch 3/5: 100% 704/704 [00:37<00:00, 18.91it/s]
Epoch 3: train_loss=0.1401 val_acc=0.9441
Epoch 4/5: 100% 704/704 [00:37<00:00, 18.95it/s]
Epoch 4: train_loss=0.1268 val_acc=0.9420
Epoch 5/5: 100% 704/704 [00:37<00:00, 18.97it/s]
Epoch 5: train_loss=0.1157 val_acc=0.9365
Saved checkpoint to checkpoints/cnn_cifake.pt


In [19]:
!python -m src.infer --input_dir data/cifake/test --output_json outputs/preds.json --checkpoint checkpoints/cnn_cifake.pt

Running inference: 100% 20000/20000 [00:30<00:00, 655.78it/s]
Wrote 20000 predictions to outputs/preds.json


In [20]:
!cat outputs/preds.json | head -20

[
  {
    "image_path": "data/cifake/test/FAKE/0 (10).jpg",
    "pred": 0.9957
  },
  {
    "image_path": "data/cifake/test/FAKE/0 (2).jpg",
    "pred": 0.9159
  },
  {
    "image_path": "data/cifake/test/FAKE/0 (3).jpg",
    "pred": 0.9456
  },
  {
    "image_path": "data/cifake/test/FAKE/0 (4).jpg",
    "pred": 0.9999
  },
  {
    "image_path": "data/cifake/test/FAKE/0 (5).jpg",
    "pred": 0.9527


In [35]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 344 bytes | 344.00 KiB/s, done.
From https://github.com/ruicatzzz/aigc-detector
   f72c91e..7e0efd3  daphne     -> origin/daphne
Updating f72c91e..7e0efd3
Fast-forward
 src/robustness_test.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [43]:
!python -m src.robustness_test


=== REAL images ===

0907 (10).jpg:
  clean              pred=0.0015
  jpeg_q30           pred=0.0020
  jpeg_q70           pred=0.0004
  blur_sigma1.0      pred=0.0345
  blur_sigma2.0      pred=0.0710
  resize_0.5x        pred=0.0417
  resize_0.25x       pred=0.0975
  noise_sigma0.05    pred=0.0022
  color_jitter       pred=0.0006
  center_crop_80     pred=0.0152

0688 (6).jpg:
  clean              pred=0.0000
  jpeg_q30           pred=0.0000
  jpeg_q70           pred=0.0000
  blur_sigma1.0      pred=0.0000
  blur_sigma2.0      pred=0.0000
  resize_0.5x        pred=0.0000
  resize_0.25x       pred=0.0000
  noise_sigma0.05    pred=0.0000
  color_jitter       pred=0.0000
  center_crop_80     pred=0.0000

0492 (10).jpg:
  clean              pred=0.0041
  jpeg_q30           pred=0.0006
  jpeg_q70           pred=0.0022
  blur_sigma1.0      pred=0.0879
  blur_sigma2.0      pred=0.1949
  resize_0.5x        pred=0.1796
  resize_0.25x       pred=0.2956
  noise_sigma0.05    pred=0.0015
  color_

In [41]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 344 bytes | 344.00 KiB/s, done.
From https://github.com/ruicatzzz/aigc-detector
   5429f22..37621ff  daphne     -> origin/daphne
Updating 5429f22..37621ff
Fast-forward
 src/eval_accuracy.py | 19 -------------------
 src/train.py         |  2 +-
 2 files changed, 1 insertion(+), 20 deletions(-)
 delete mode 100644 src/eval_accuracy.py


In [42]:
!python -m src.train --data_dir data/cifake/train --epochs 10 --out checkpoints/cnn_cifake.pt

Using device: cuda
Classes found: {'FAKE': 0, 'REAL': 1}  (expect FAKE and REAL keys)
Epoch 1/10: 100% 352/352 [00:45<00:00,  7.68it/s]
Epoch 1: train_loss=0.3736 val_acc=0.8802 <- new best, saving
Epoch 2/10: 100% 352/352 [00:46<00:00,  7.62it/s]
Epoch 2: train_loss=0.2765 val_acc=0.9065 <- new best, saving
Epoch 3/10: 100% 352/352 [00:47<00:00,  7.45it/s]
Epoch 3: train_loss=0.2513 val_acc=0.9382 <- new best, saving
Epoch 4/10: 100% 352/352 [00:48<00:00,  7.31it/s]
Epoch 4: train_loss=0.2273 val_acc=0.9296
Epoch 5/10: 100% 352/352 [00:46<00:00,  7.52it/s]
Epoch 5: train_loss=0.2104 val_acc=0.9399 <- new best, saving
Epoch 6/10: 100% 352/352 [00:46<00:00,  7.62it/s]
Epoch 6: train_loss=0.1974 val_acc=0.9452 <- new best, saving
Epoch 7/10: 100% 352/352 [00:46<00:00,  7.58it/s]
Epoch 7: train_loss=0.1827 val_acc=0.9541 <- new best, saving
Epoch 8/10: 100% 352/352 [00:46<00:00,  7.53it/s]
Epoch 8: train_loss=0.1714 val_acc=0.9512
Epoch 9/10: 100% 352/352 [00:46<00:00,  7.53it/s]
Epoch 9:

In [3]:
from datasets import load_dataset
from pathlib import Path
import os

N_SAMPLES = 10000  # adjust based on how much you want

out_dir = Path("data/sid_subset")
(out_dir / "REAL").mkdir(parents=True, exist_ok=True)
(out_dir / "FAKE").mkdir(parents=True, exist_ok=True)

ds = load_dataset("saberzl/SID_Set", split="train", streaming=True)

real_count, fake_count = 0, 0
for i, example in enumerate(ds):
    if i >= N_SAMPLES:
        break
    label = example["label"]  # 0=real, 1=full_synthetic, 2=tampered
    img = example["image"]    # PIL Image directly

    if label == 0:
        img.convert("RGB").save(out_dir / "REAL" / f"{example['img_id']}.jpg")
        real_count += 1
    else:  # both full_synthetic and tampered count as AIGC/FAKE for this task
        img.convert("RGB").save(out_dir / "FAKE" / f"{example['img_id']}.jpg")
        fake_count += 1

print(f"Saved {real_count} REAL, {fake_count} FAKE images to {out_dir}")

README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/249 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/249 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

Saved 3338 REAL, 6662 FAKE images to data/sid_subset


In [ ]:
!python -m src.train --data_dir data/sid_subset --epochs 10 --out checkpoints/cnn_sid.pt

Using device: cuda
Classes found: {'FAKE': 0, 'REAL': 1}  (expect FAKE and REAL keys)
Epoch 1/10: 100% 36/36 [04:37<00:00,  7.72s/it]
Epoch 1: train_loss=0.5929 val_acc=0.7000 <- new best, saving
Epoch 2/10:  50% 18/36 [02:22<02:03,  6.87s/it]